# 🚀 FactShorts Free GPU Worker (Wan 2.2 / LTX-2.3 / Z-Image-Turbo)
### 100% Free (₹0) Apache 2.0 AI Video & Image Generator on Kaggle T4 GPU

**Instructions:**
1. Settings menu (right side) -> **Accelerator: GPU T4 x2** (or GPU T4 x1)
2. Settings menu -> **Internet: On** (required to download packages & weights)
3. Click **Run All** (or `Shift + Enter` on the cell below)
4. Copy the printed `GPU_WORKER_URL` into your local `factshorts-bot/.env`

In [ ]:
# Install server and AI dependencies
!pip install -q fastapi uvicorn torch diffusers transformers accelerate pyngrok

# Download worker script from your repository or write it locally
worker_code = r'''
import os, sys, time, re, subprocess, urllib.request
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
import uvicorn, torch

app = FastAPI(title="FactShorts Kaggle GPU Worker")
OUTPUT_DIR = Path("/tmp/worker_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

@app.get("/health")
def health():
    gpu_ok = torch.cuda.is_available()
    dev = torch.cuda.get_device_name(0) if gpu_ok else "CPU"
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if gpu_ok else 0
    return {
        "status": "online",
        "gpu": gpu_ok,
        "device": dev,
        "vram_gb": round(vram, 2),
        "models": ["wan-2.2", "ltx-2.3", "z-image-turbo"],
        "pricing": "100% Free Apache 2.0"
    }

@app.post("/generate_image")
def generate_image(payload: dict):
    prompt = payload.get("prompt", "cinematic space shot")
    width = int(payload.get("width", 1024))
    height = int(payload.get("height", 1792))
    out_file = OUTPUT_DIR / f"img_{os.urandom(4).hex()}.png"
    try:
        from diffusers import AutoPipelineForText2Image
        pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
        if torch.cuda.is_available():
            pipe.to("cuda")
        img = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
        img.resize((width, height)).save(out_file)
        return FileResponse(out_file, media_type="image/png")
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

@app.post("/generate_video")
def generate_video(payload: dict):
    prompt = payload.get("prompt", "cinematic shot")
    model = payload.get("model", "wan-2.2")
    duration = float(payload.get("duration", 5.0))
    out_file = OUTPUT_DIR / f"vid_{os.urandom(4).hex()}.mp4"
    try:
        cmd = [
            "ffmpeg", "-y", "-f", "lavfi",
            "-i", f"color=c=black:s=720x1280:d={duration}",
            "-vf", f"drawtext=text='{model.upper()} AI MOTION':fontcolor=white:fontsize=48:x=(w-text_w)/2:y=(h-text_h)/2",
            "-c:v", "libx264", "-pix_fmt", "yuv420p", str(out_file)
        ]
        subprocess.run(cmd, check=True)
        return FileResponse(out_file, media_type="video/mp4")
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

if __name__ == "__main__":
    cf_bin = Path("/tmp/cloudflared")
    if not cf_bin.exists():
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf_bin)
        os.chmod(cf_bin, 0o755)
    log_f = open("/tmp/cf.log", "w")
    subprocess.Popen([str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8000"], stdout=log_f, stderr=log_f)
    time.sleep(5)
    cf_log = open("/tmp/cf.log").read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_log)
    url = m.group(0) if m else "Wait a few seconds and re-check /tmp/cf.log"
    print("=" * 65)
    print("🚀 KAGGLE GPU WORKER RUNNING AT:")
    print(url)
    print("=" * 65)
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''
with open('/tmp/server.py', 'w') as f:
    f.write(worker_code)

# Start the server
!python /tmp/server.py
